# 71) One-Way MANOVA Nedir?
ANOVA'da (Konu 65-66), **tek bir bağımlı değişkeni** (örn: sadece "Memnuniyet"), tek bir kategorik faktöre göre karşılaştırıyorduk. MANOVA (Multivariate ANOVA), **birden fazla bağımlı değişkeni AYNI ANDA**, tek bir kategorik faktöre göre karşılaştırır.

## Neden Ayrı Ayrı ANOVA'lar Yerine MANOVA?
Diyelim ki "Memnuniyet" VE "Sadakat Puanı" diye 2 bağımlı değişkenimiz var, ve bunları 4 farklı kanala göre karşılaştırmak istiyorsun. İlk akla gelen "kolay" yol: 2 ayrı One-Way ANOVA çalıştırmak (biri Memnuniyet için, biri Sadakat için). Ama bu yaklaşımın 2 sorunu var:
1. **Alpha Inflation** (Konu 64) — 2 ayrı test, riski şişirir.
2. **Değişkenler arası ilişkiyi göz ardı eder** — Memnuniyet ve Sadakat muhtemelen birbiriyle **ilişkili** (Konu'daki Varyans-Kovaryans Matrisi fikri tam burada işe yarıyor) — MANOVA bu ilişkiyi hesaba katarak, ayrı ayrı bakmanın kaçıracağı **birleşik/kombine** farkları yakalayabilir

## Hipotezler
- **H0:** Gruplar arasında, bağımlı değişkenlerin (Memnuniyet + Sadakat 
  birlikte) **kombinasyonunda** hiçbir fark yoktur
- **H1:** En az bir grup, bağımlı değişkenlerin bu kombinasyonunda 
  farklıdır

## Test İstatistiği
MANOVA, tek bir F yerine, genelde **Wilks' Lambda** (ya da Pillai's 
Trace, Hotelling's Trace gibi alternatifler) adı verilen bir istatistik 
kullanır — bunlar, Varyans-Kovaryans Matrisi'nin (bir önceki konumuz) 
matematiksel özelliklerinden (determinant, özdeğerler gibi) türetilir. 
Detay formüllerine girmiyoruz, sadece "ANOVA'nın F'inin, çok değişkenli 
versiyonu" olduğunu bilmek yeterli.

## Python'da Kullanımı
```python
from statsmodels.multivariate.manova import MANOVA

manova_model = MANOVA.from_formula('Memnuniyet + Sadakat ~ C(Kanal)', data=df)
print(manova_model.mv_test())
```
## MANOVA'nın Varsayımları
MANOVA, ANOVA'nın varsayımlarının (Konu 68) **çok değişkenli genişletilmiş 
hali**ni gerektirir:

1. **Çok Değişkenli Normallik (Multivariate Normality):** Bağımlı 
   değişkenlerin **her biri ayrı ayrı** değil, **birlikte/kombinasyon 
   halinde** çok değişkenli normal dağılıma uyması gerekir. Tek tek 
   Shapiro-Wilk yeterli değildir — ama pratikte, her değişkenin ayrı ayrı 
   normal olması, iyi bir ön işarettir. Resmi test için Mardia's Test 
   kullanılabilir (ileri seviye).

2. **Varyans-Kovaryans Matrislerinin Homojenliği (Box's M Testi):** 
   Konu'daki Levene testinin çok değişkenli versiyonu — her grubun 
   **Varyans-Kovaryans Matrisi'nin** (bir önceki konumuz) birbirine 
   yakın/homojen olması gerekir. **Box's M Testi** ile kontrol edilir.
   - H0: Grupların kovaryans matrisleri eşittir (homojendir)
   - Python: `pingouin` kütüphanesinde `pg.box_m()` fonksiyonu

3. **Bağımsız Gözlemler:** Gruplar birbirinden bağımsız olmalı (One-Way 
   ANOVA'daki gibi)

4. **Doğrusal İlişki (Linearity):** Bağımlı değişkenler kendi aralarında 
   **doğrusal bir ilişki** içinde olmalı (aşırı doğrusal olmayan 
   ilişkiler MANOVA'nın gücünü azaltır) — Korelasyon bölümünde (Konu 89) 
   bunu nasıl kontrol edeceğimizi göreceğiz.

5. **Çoklu Doğrusal Bağlantı Olmaması (No Multicollinearity):** Bağımlı 
   değişkenler birbirleriyle **çok fazla** ilişkili (neredeyse aynı şeyi 
   ölçüyor) olmamalı — aşırı yüksek korelasyon, matrisin matematiksel 
   olarak "tekil/singular" olmasına yol açabilir, bu da testi bozar.

## Box's M Testinin Özel Bir Notu
Box's M, **çok hassas** bir testtir — küçük sapmalar bile kolayca 
"anlamlı" (H0 reddi) çıkabilir, özellikle büyük örneklemlerde. Bu yüzden 
sıkı bir α (örn: 0.001) kullanmak literatürde yaygın bir pratiktir, 
standart 0.05 çok katı sonuçlar verebilir.

## MANOVA Sonrası Ne Yapılır?
MANOVA da (ANOVA gibi) sadece "en az bir yerde fark var" der, **hangi bağımlı değişkende, hangi gruplar arasında** farkın olduğunu söylemez. Bu yüzden MANOVA anlamlı çıkarsa, genelde **her bağımlı değişken için ayrı ayrı** ANOVA + Post-Hoc testlerle takip edilir (artık Alpha Inflation endişesi daha az, çünkü MANOVA zaten "genel bir fark var" onayını verdi).

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 30

kanal_ortalamalari = {
    'Telefon': {'memnuniyet': 60, 'sadakat': 50},
    'Canlı Sohbet': {'memnuniyet': 85, 'sadakat': 51},
    'Email': {'memnuniyet': 70, 'sadakat': 49},
}

kanallar, memnuniyet, sadakat = [], [], []
for kanal, ort in kanal_ortalamalari.items():
    kanallar += [kanal]*n
    memnuniyet += list(np.random.normal(ort['memnuniyet'], 8, n))
    sadakat += list(np.random.normal(ort['sadakat'], 8, n))

df = pd.DataFrame({'Kanal': kanallar, 'Memnuniyet': memnuniyet, 'Sadakat': sadakat})

from statsmodels.multivariate.manova import MANOVA
manova_model = MANOVA.from_formula('Memnuniyet + Sadakat ~ C(Kanal)', data=df)
print(manova_model.mv_test())

                   Multivariate linear model
                                                               
---------------------------------------------------------------
       Intercept         Value  Num DF  Den DF  F Value  Pr > F
---------------------------------------------------------------
          Wilks' lambda  0.0155 2.0000 86.0000 2729.7689 0.0000
         Pillai's trace  0.9845 2.0000 86.0000 2729.7689 0.0000
 Hotelling-Lawley trace 63.4830 2.0000 86.0000 2729.7689 0.0000
    Roy's greatest root 63.4830 2.0000 86.0000 2729.7689 0.0000
---------------------------------------------------------------
                                                               
---------------------------------------------------------------
          C(Kanal)        Value  Num DF  Den DF  F Value Pr > F
---------------------------------------------------------------
            Wilks' lambda 0.3201 4.0000 172.0000 33.0045 0.0000
           Pillai's trace 0.6858 4.0000 174.0000 22.6987 0.

### Sonuç
Öncelikle kanallar arasında, Memnuniyet ve Sadakat kombinasyonunda fark olup olmadığını test etmek için Tek Yönlü MANOVA analizi uyguladık. MANOVA sonuçlarına göre, kanallar arasında bu kombinasyonda istatistiksel olarak anlamlı bir fark bulunmuştur (Wilks' Lambda=0.32, F=33.00, p<0.001). Ancak MANOVA bize sadece genel bir farkın var olduğunu söyledi, bu farkın hangi değişkenden (Memnuniyet mi, Sadakat mi) kaynaklandığını belirtmedi. Bu nedenle her bağımlı değişken için ayrı ayrı One-Way ANOVA uyguladık. Sonuçlara göre, kanallar arasındaki fark Memnuniyet değişkeninde son derece anlamlı çıkarken (F=88.63, p<0.001), Sadakat değişkeninde anlamlı bir fark bulunamamıştır (F=0.75, p=0.477). Bu doğrultuda, MANOVA'nın tespit ettiği genel farkın tamamen Memnuniyet kaynaklı olduğu, farklı destek kanallarının müşteri sadakatini anlamlı şekilde etkilemediği söylenebilir.